# 提示词
发送给大模型的所有消息都可以称为提示词（Prompt），它直接影响模型的输出结果。

其中，SystemMessage尤为重要，我们把SystemMessage称为**系统提示词（System Prompt）**，它可以给模型设定角色和本次聊天的背景，对模型生成的内容有很大的影响。

## 系统提示词
在创建智能体时，可以直接设定 system prompt，不必在每次发送消息时指定


没有设定系统提示词：

In [ ]:
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat"
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode = "messages"
):
    print(token.content, end="", flush=True)

设定了海盗这个角色：

In [ ]:
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt = "像海盗一样说话。"
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode = "messages"
):
    print(token.content, end="", flush=True)

## 提示词工程
通过优化System Prompt从而让模型输出更理想的结果的这一过程，我们称为提示词工程（Prompt Engineering）。

也就是说，提示词优化不是一锤子买卖，而是一个不断优化、测试、再优化的过程。那么，提示词到底该怎么写呢？

从内容来说，提示词通常包含以下几个部分，通常按此顺序排列：

- 身份（Identity）：描述AI的职责、沟通风格和总体目标。
- 说明（Instructions）：请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？
- 示例（Examples）：提供可能的输入示例，以及模型期望的输出。
- 背景信息（Context）：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。

从格式来说，在编写System Prompt时，您可以使用Markdown格式和XML 标签的组合来帮助模型理解提示和上下文数据的逻辑边界。

- Markdown 的标题和列表有助于标记提示的不同部分，并向模型传达层级结构。它们还可以提高开发过程中提示的可读性。
- XML 标签可以帮助明确区分一段内容（例如用作参考的辅助文档、对话示例等）的起始和结束位置。

示例：
```
# Identity :定义模型身份与人物边界

You are a helpful assistant that labels short product reviews as Positive, Negative, or Neutral.

# Instructions :定义执行规则与输出约束

* Only output a single word in your response with no additional formatting or commentary.
    -- 你的回复中仅输出一个单词，不包含任何额外的格式或评论
* Your response should only be one of the words
   "Positive", "Negative", or "Neutral" depending on the sentiment of the product review you are given.
   --根据给定的产品评论所表达的情感倾向，你的回复必须且只能是以下三个单词之一：“Positive”、“Negative”或“Neutral”

# Examples :少样本示例

<product_review id="example-1">
I absolutely love this headphones — sound quality is amazing! --我简直太喜欢这款耳机了——音质棒极了！
</product_review>

<assistant_response id="example-1">
Positive
</assistant_response>

<product_review id="example-2">
Battery life is okay, but the ear pads feel cheap. --电池续航还行，但耳垫摸起来感觉很廉价。
</product_review>

<assistant_response id="example-2">
Neutral
</assistant_response>

<product_review id="example-3">
Terrible customer service, I'll never buy from them again. --糟糕的客户服务，我再也不会从他们那里购买了。
</product_review>

<assistant_response id="example-3">
Negative
</assistant_response>
```

### 设定角色和详细指令

**角色**可以帮助模型认清自己的身份，以对应的身份来回答问题。

**指令**则告诉模型需要遵循哪些规则，应该做什么，不应该做什么



In [ ]:
system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写 Python 代码。

# 指令
- 定义变量时，使用 snake case 命名法，而不是 camel case 命名法。
- 不要反悔 markdown 格式说明，仅仅返回代码即可。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt = system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="请帮我定义一个变量，表示我的名字，例如“heima”。")]},
    stream_mode = "messages"
):
    print(token.content, end="", flush=True)

### Few-Shot examples

通过举例的方式让模型学习例子来回答

In [ ]:
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="地球的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

### 结构化输出
通过在提示词中明确要求模型输出特定格式的内容，可以让模型输出更规范、更易于解析的结果。

In [ ]:
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="太阳的首都是什么?")]},
)

print(response['messages'][-1].content)

在LangChain中，实现结构化输出会更加简单。我们无需自己在提示词中添加描述实现结构化输出，而仅仅是设定好一个数据类型即可。

首先，定义一个类，用来封装模型要输出的数据：

In [ ]:
from pydantic import BaseModel
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

然后，就可以在创建 Agent 时，设定好输出格式

In [ ]:
agent = create_agent(
    model = "deepseek-chat",
    system_prompt = "你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format = CapitalInfo
)

response = agent.invoke(
    {"messages": [HumanMessage(content="三体的首都是什么？")]}
)

# print(response)
"""
{'messages': [HumanMessage(content='月球的首都是什么？', additional_kwargs={}, response_metadata={}, id='8199d6b9-924b-4fa5-8019-859ec6918d3d'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 134, 'prompt_tokens': 355, 'total_tokens': 489, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 320}, 'prompt_cache_hit_tokens': 320, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'd356c819-079e-4f72-9ce9-df14f38f7872', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9116-79f7-7341-98aa-160c2cd27063-0', tool_calls=[{'name': 'CapitalInfo', 'args': {'name': '月宫', 'location': '月球雨海区域，靠近阿波罗17号着陆点', 'vibe': '未来主义与古典融合，低重力下的优雅建筑，透明穹顶下的花园城市', 'economy': '氦-3开采与加工，月球旅游，科学研究，低重力制造，太空港服务'}, 'id': 'call_00_XfpDuZ8yv2BaJthvJ0U1ASG6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 355, 'output_tokens': 134, 'total_tokens': 489, 'input_token_details': {'cache_read': 320}, 'output_token_details': {}}), ToolMessage(content="Returning structured response: name='月宫' location='月球雨海区域，靠近阿波罗17号着陆点' vibe='未来主义与古典融合，低重力下的优雅建筑，透明穹顶下的花园城市' economy='氦-3开采与加工，月球旅游，科学研究，低重力制造，太空港服务'", name='CapitalInfo', id='c1ec05bb-65cf-4b92-8cc9-6405d4938370', tool_call_id='call_00_XfpDuZ8yv2BaJthvJ0U1ASG6')], 'structured_response': CapitalInfo(name='月宫', location='月球雨海区域，靠近阿波罗17号着陆点', vibe='未来主义与古典融合，低重力下的优雅建筑，透明穹顶下的花园城市', economy='氦-3开采与加工，月球旅游，科学研究，低重力制造，太空港服务')}
"""

city = response['structured_response']
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")